# Module 3: Class Imbalance, SMOTE Interpolation & Ensemble Learning

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Precision/Recall/F1 threshold tuning, SMOTE interpolation, Random Forests, and from-scratch AdaBoost.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM03_Class_Imbalance_Ensembles.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. Why Accuracy Fails on Rare Classes & Threshold Tuning

On a 90/10 imbalanced crop-failure dataset, predicting `0` everywhere scores 90% accuracy with 0% recall. Precision, Recall, F1, and MCC measure performance on the rare class.

In [1]:
import math
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split


def binary_metrics(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    acc = (tp + tn) / len(y_true)
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    f1 = 2 * prec * rec / max(1e-12, prec + rec)
    return acc, prec, rec, f1


rng = np.random.default_rng(3003)
X = np.vstack([rng.normal(0.0, 1.0, (900, 5)), rng.normal(1.6, 0.9, (100, 5))])
y = np.array([0] * 900 + [1] * 100)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=7)

acc0, _, rec0, f10 = binary_metrics(y_te, np.zeros_like(y_te))
print(f"Lazy baseline -> accuracy={acc0:.3f}, recall={rec0:.3f}, F1={f10:.3f}")

rf = RandomForestClassifier(n_estimators=60, class_weight="balanced", random_state=7).fit(X_tr, y_tr)
probs = rf.predict_proba(X_te)[:, 1]
for thr in (0.5, 0.3, 0.2):
    acc, prec, rec, f1 = binary_metrics(y_te, (probs >= thr).astype(int))
    print(f"RF threshold={thr:.1f} -> acc={acc:.3f}, prec={prec:.3f}, rec={rec:.3f}, F1={f1:.3f}")


Lazy baseline -> accuracy=0.900, recall=0.000, F1=0.000
RF threshold=0.5 -> acc=0.960, prec=0.846, rec=0.733, F1=0.786
RF threshold=0.3 -> acc=0.957, prec=0.743, rec=0.867, F1=0.800
RF threshold=0.2 -> acc=0.940, prec=0.650, rec=0.867, F1=0.743


## 2. SMOTE-Style Minority Interpolation & Decision Stump AdaBoost

Bagging and Random Forests reduce variance across bootstrap trees; Boosting fits sequential weak learners concentrating weight on misclassified records.

In [2]:
from sklearn.tree import DecisionTreeClassifier


def smote_interpolate(X_min, n_new, k=5, seed=7):
    rng = np.random.default_rng(seed)
    out = np.zeros((n_new, X_min.shape[1]))
    for idx in range(n_new):
        i = int(rng.integers(0, len(X_min)))
        nbrs = np.argsort(np.linalg.norm(X_min - X_min[i], axis=1))[1 : k + 1]
        j = int(rng.choice(nbrs))
        out[idx] = X_min[i] + rng.random() * (X_min[j] - X_min[i])
    return out


X_syn = smote_interpolate(X_tr[y_tr == 1], n_new=300)
X_aug = np.vstack([X_tr, X_syn])
y_aug = np.concatenate([y_tr, np.ones(300, dtype=int)])

# Train AdaBoost from scratch on augmented data
n_aug = len(y_aug)
y_pm = np.where(y_aug == 1, 1.0, -1.0)
w = np.full(n_aug, 1.0 / n_aug)
stumps = []
for _ in range(25):
    st = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_aug, y_aug, sample_weight=w)
    pr = np.where(st.predict(X_aug) == 1, 1.0, -1.0)
    err = min(max(float(np.sum(w[pr != y_pm])), 1e-8), 1.0 - 1e-8)
    alpha = 0.5 * math.log((1.0 - err) / err)
    w *= np.exp(-alpha * y_pm * pr)
    w /= w.sum()
    stumps.append((st, alpha))

score = sum(a * np.where(st.predict(X_te) == 1, 1.0, -1.0) for st, a in stumps)
acc, prec, rec, f1 = binary_metrics(y_te, (score >= 0).astype(int))
print(f"From-scratch AdaBoost + SMOTE -> prec={prec:.3f}, rec={rec:.3f}, F1={f1:.3f}")
assert f1 > 0.70


From-scratch AdaBoost + SMOTE -> prec=0.730, rec=0.900, F1=0.806
